# Understand a delivery RL environment

Run from the repository or notebooks folder. Start by watching one decision; then inspect how the learner scores it. Synthetic data only.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'dispatchlab').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from dispatchlab import DispatchEnv, Config
from dispatchlab.agent import LinearQAgent, train_episode
from dispatchlab.policies import heuristic
from dispatchlab.environment import ACTION_NAMES


## 1. What can the vehicle see?
An observation contains revealed orders and current vehicle state. Future arrivals are not visible.

In [ ]:
env = DispatchEnv()
obs, info = env.reset(seed=200001)
obs

In [ ]:
[(ACTION_NAMES[i], bool(allowed)) for i, allowed in enumerate(info['action_mask'])]

## 2. What changes if we wait?
A step advances time, reveals any new order, and updates costs. It does not necessarily produce a positive reward.

In [ ]:
nxt, reward, terminated, truncated, info = env.step(0)
print('Reward:', reward)
print(nxt)

## 3. Run a simple rule for one shift
Understand earliest-deadline dispatch before evaluating the learned policy.

In [ ]:
obs, info = env.reset(seed=200001)
done = False
while not done:
    action = heuristic(obs, info['action_mask'], 'deadline')
    obs, reward, done, _, info = env.step(action)
print(env.metrics)

## 4. Inspect a trained agent
These are estimated action values, not probabilities. Features are readable; their weights were learned.

In [ ]:
agent = LinearQAgent.load(ROOT / 'models' / 'linear_q.json')
obs, info = env.reset(seed=200001)
{ACTION_NAMES[a]: round(q, 2) for a, q in agent.values(obs, info['action_mask']).items()}

## 5. Compare on the same offered demand
Use identical seeds. Include rejected and unfinished requests in your interpretation.

In [ ]:
from dispatchlab.experiment import run_episode
for name, policy in [('Earliest deadline', lambda o,m: heuristic(o,m)), ('Learned policy', agent.act)]:
    metrics, _ = run_episode(Config(), 200001, policy)
    print(name, {k: round(metrics[k], 3) for k in ['reward', 'on_time_rate', 'completion_rate']})

## 6. Try a tiny learning run
This only illustrates the update loop. Fifty episodes are not a reliable benchmark. The bundled models trained on 3,000 episodes each.

In [ ]:
beginner = LinearQAgent(seed=1)
rewards = [train_episode(DispatchEnv(), beginner, seed, epsilon=0.4) for seed in range(50)]
print('First five rewards:', rewards[:5])
print('Learned weights:', beginner.weights.round(2))

## Questions to answer

- Why might waiting now improve reward later?
- Why can a policy with fewer travel ticks still have worse service?
- Why must terminal states have no bootstrapped future value?
- What would change if travel durations were uncertain?

Read `docs/LEARNING_GUIDE.md` next. Do not tune on the bundled test set after inspecting its results.